# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kuteesatendojeremiah/Tendojerry-Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb huggingface_hub

import os, getpass
import duckdb

# CI and power users set HF_TOKEN in the environment or Colab Secrets (🔑 icon);
# everyone else gets the safe interactive prompt. Never hardcode the token in a cell — this repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:>12,} rows")


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. **One row** = one content item, for one client, on one day.
2. **Table used:** `fact_content_daily_performance` (joined to `dim_clients` only to read each client's history-start dates — never for features or labels).
3. **Time window:** the mid-panel month **`2026-03`** (March 2026), with the 30 days immediately before it (Jan 30 – Feb 28, "prev30") as the comparison period. The panel's sealed final month (June 2026, `fact_content_daily_performance_sample`) is never touched here — it's the natural outcome window of any past→future label.

In [ ]:
# Mid-panel month for this notebook — never the sealed final month (June 2026).
MONTH_START = "2026-03-01"
MONTH_END_EXCL = "2026-04-01"      # half-open: report_date < MONTH_END_EXCL
PREV30_START = "2026-01-30"        # the 30 days immediately before MONTH_START
PREV30_END_EXCL = MONTH_START

print(f"Iterating on month={MONTH_START[:7]} | prev30 window: [{PREV30_START}, {PREV30_END_EXCL})")


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

4. **What I'd rank (label/proxy):** `is_declining` — a page's March impressions falling more than 20% below its prev30 (Feb) impressions. It's a proxy for "needs attention," defined by a rule, not an observed editorial judgment — and it's built only from March data that the five features below never see.
5. **One thing I deliberately exclude:** `access_profile` (and similar account/product metadata on `dim_clients`) — it describes FlyRank's commercial relationship with the client, not the page's observed search performance, so it has no business being a ranking signal.

In [ ]:
# No verification query needed here — the three required proof queries live in Section 3.
# This cell exists only so "Run All" has a code cell under every markdown answer.
EXCLUDED_FIELDS = ["access_profile"]  # FlyRank account metadata, not observed search performance
print("Excluded from any feature set:", EXCLUDED_FIELDS)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three facts about this slice, each proven with exactly one query on `month=2026-03`:

1. **Grain** — one row really is one content item, for one client, on one day.
2. **Slice size + date span** — how many rows this month's slice has, and that its dates match March 2026 exactly.
3. **Availability** — how many rows have GA4 engagement data actually available (`ga4_data_available IS TRUE`).

Then: a five-feature frame built from the prev30 window, and the one deliberate leak.

In [ ]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END_EXCL}'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Fact 1 — grain violations in month=2026-03 (should be empty):")
print(grain_check)


### Fact 2 — slice size and date span

In [ ]:
slice_summary = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END_EXCL}'
""").df()
print("Fact 2 — March 2026 slice size and date span:")
print(slice_summary)


### Fact 3 — availability (`IS TRUE`)

In [ ]:
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END_EXCL}'
""").df()
availability["pct_available"] = (availability["ga4_available_rows"] / availability["total_rows"] * 100).round(1)
print("Fact 3 — GA4 availability this month (IS TRUE filter):")
print(availability)


### Five features (max), from the prev30 window (Jan 30 – Feb 28)

Every feature below is computed only from data before `MONTH_START` — so each one is
knowable at the moment the ranking decision would actually be made, before March's outcome
exists.

1. **`imp_prev30`** — Feb impressions, summed. Available when: fully observed the moment Feb ends, before March begins.
2. **`clk_prev30`** — Feb clicks, summed. Available when: same trailing count, nothing from March.
3. **`avg_position_prev30`** — Feb average GSC position. Available when: computed entirely from Feb's daily rows.
4. **`ctr_prev30`** — Feb clicks ÷ Feb impressions. Available when: a ratio of two prev30 counts only.
5. **`active_days_prev30`** — count of Feb days with any impressions. Available when: a tally of past days only, no future information.

In [ ]:
feature_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_prev30,
           SUM(gsc_clicks) AS clk_prev30,
           AVG(gsc_avg_position) AS avg_position_prev30,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_prev30
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{PREV30_START}' AND report_date < DATE '{PREV30_END_EXCL}'
    GROUP BY 1, 2
""").df()
feature_frame["ctr_prev30"] = feature_frame["clk_prev30"] / feature_frame["imp_prev30"]

print(f"Feature frame: {len(feature_frame):,} content items x 5 features")
feature_frame.head()


### The trap: one label-derived column, on purpose

The label — did March impressions drop more than 20% vs prev30 — comes from March data the
five features above never touch. Here I add ONE column built straight from that same March
data, watch a quick classifier's score jump toward perfect, then delete the column and report
the honest number.

In [ ]:
march = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_march
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END_EXCL}'
    GROUP BY 1, 2
""").df()

data = feature_frame.merge(march, on=["client_hash_id", "content_hash_id"], how="inner")
data = data[data["imp_prev30"] > 0].copy()
data["is_declining"] = (data["imp_march"] < 0.8 * data["imp_prev30"]).astype(int)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["imp_prev30", "clk_prev30", "avg_position_prev30", "ctr_prev30", "active_days_prev30"]

def quick_auc(feature_cols, df):
    X = df[feature_cols].fillna(0)
    y = df["is_declining"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

print(f"Honest AUC (5 prev30 features only):        {quick_auc(honest_features, data):.3f}")

# --- THE TRAP: add a column built straight from the label's own March data ---
data["imp_march_leak"] = data["imp_march"]  # deliberately re-adding the outcome as a "feature"
leaky_features = honest_features + ["imp_march_leak"]
print(f"Leaky AUC (+ raw March impressions as a feature): {quick_auc(leaky_features, data):.3f}")

# --- remove the leak, keep only the honest features going forward ---
data = data.drop(columns=["imp_march_leak"])
print(f"Honest AUC restored (leak column deleted):  {quick_auc(honest_features, data):.3f}")


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**One named limitation:** GA4 engagement data is not available for every row in this slice (Fact 3 above) — rows before a client's `ga4_data_start` are zero-filled with `ga4_data_available = FALSE`, not missing at random. So any feature built from engagement columns without the `IS TRUE` filter would silently treat "not onboarded yet" as "zero engagement," which March 2026 alone cannot distinguish from a real collapse.

In [ ]:
# The limitation above is the same evidence as Fact 3 — restated here, not re-queried.
print("Limitation evidenced in Fact 3 — GA4 availability, month=2026-03:")
print(availability)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.